<a href="https://colab.research.google.com/github/Sherzod19s/ECON3086-Python-Programming-for-FinTech/blob/main/ex9b_more_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 9b - More operations with dataframe

This exercise is to find the Labour Force Participation rate (LFPR) for a list of 5 countries, leveraging some data from nasdaq-data-link

LFPR = Labour Force / Population

However, the Labour Force data are not available through nasdaq-data-link, so we are going to calculate it by Employment Popluation and Unemployment rate.

Employment Rate = Employment Population / Labour Force
Labour Force = Employment Population / Employment Rate
Labour Force = Employment Population / (1 - Unemployment Rate)


In [9]:
import nasdaqdatalink
import pandas as pd

nasdaqdatalink.ApiConfig.api_key = "x9M_pZutNNPnha1WDdjZ"

def get_imf_data(indicator, countries_list, from_year, to_year):
    indicator_string = ["{}_{}".format(country_code,indicator) for country_code in countries_list]
    df= nasdaqdatalink.get_table('QDL/ODA',
                                    date = {'gte': '{}-01-01'.format(from_year),
                                            'lte': '{}-12-31'.format(to_year)},
                                    indicator=indicator_string)
    # Split the indicator column into country_code
    df['country_code'] = df['indicator'].str.split('_').str[0]
    # Remove the indicator column
    df = df.drop(columns=['indicator'])
    df = df.rename(columns={'value': indicator})
    return df


# LE - Employment, Millions
# LP - Population, Millions
# LUR - Unemployment Rate, Millions
employment_df = get_imf_data('LE', ['AUS', 'CAN', "USA", 'JPN', 'GBR', 'FRA'], 1999, 2024)
population_df = get_imf_data('LP', ['AUS', 'CAN', "USA", 'JPN', 'GBR', 'FRA'], 1999, 2024)
unemployment_rate_df = get_imf_data('LUR', ['AUS', 'CAN', "USA", 'JPN', 'GBR', 'FRA'], 1999, 2024)

In [10]:
employment_df

,date,LE,country_code
None,,,
0,2024-12-31,159.103,USA
1,2023-12-31,159.375,USA
2,2022-12-31,158.297,USA
3,2021-12-31,152.586,USA
4,2020-12-31,147.813,USA
...,...,...,...
151,2003-12-31,9.392,AUS
152,2002-12-31,9.185,AUS
153,2001-12-31,9.016,AUS


In [11]:
population_df

,date,LP,country_code
None,,,
0,2024-12-31,337.765,USA
1,2023-12-31,335.537,USA
2,2022-12-31,333.530,USA
3,2021-12-31,332.314,USA
4,2020-12-31,331.257,USA
...,...,...,...
151,2003-12-31,19.827,AUS
152,2002-12-31,19.605,AUS
153,2001-12-31,19.386,AUS


In [12]:
unemployment_rate_df

,date,LUR,country_code
None,,,
0,2024-12-31,4.920,USA
1,2023-12-31,3.832,USA
2,2022-12-31,3.642,USA
3,2021-12-31,5.367,USA
4,2020-12-31,8.092,USA
...,...,...,...
151,2003-12-31,5.942,AUS
152,2002-12-31,6.358,AUS
153,2001-12-31,6.775,AUS


Q1. Find the highest unemployment rate between 1999 for 2024 for each country, in decending order. (tips: using groupBy)

In [14]:
unemployment_rate_df.groupby('country_code')['LUR'].max().sort_values(ascending=False)

,LUR
country_code,
FRA,10.442
CAN,9.725
USA,9.608
GBR,8.100
AUS,6.867
JPN,5.358


Q2. Find the lowest unemployment rate between 1999 for 2024 for each country, in acending order

In [16]:
unemployment_rate_df.groupby('country_code')['LUR'].min().sort_values(ascending=True)

,LUR
country_code,
JPN,2.300
USA,3.642
GBR,3.700
AUS,3.708
CAN,5.275
FRA,7.268


Q3. Join all the dataframe on country and year, so that each row in this dataframe contain the value of employment, population and unemployment rate and the corresponding country and year.

In [18]:
combined_df = pd.merge(employment_df, population_df, on=['date', 'country_code'])
combined_df = pd.merge(combined_df, unemployment_rate_df, on=['date', 'country_code'])
combined_df

,date,LE,country_code,LP,LUR
0,2024-12-31,159.103,USA,337.765,4.920
1,2023-12-31,159.375,USA,335.537,3.832
2,2022-12-31,158.297,USA,333.530,3.642
3,2021-12-31,152.586,USA,332.314,5.367
4,2020-12-31,147.813,USA,331.257,8.092
...,...,...,...,...,...
151,2003-12-31,9.392,AUS,19.827,5.942
152,2002-12-31,9.185,AUS,19.605,6.358
153,2001-12-31,9.016,AUS,19.386,6.775
154,2000-12-31,8.899,AUS,19.141,6.292


Q4. Create a new column "labour_force"

    Labour force = Employment / (1 - Unemployment Rate/100)
    This represents the total population that is either employed or actively looking for work

In [19]:
combined_df['labour_force'] = combined_df['LE']/(1-combined_df['LUR']/100)
combined_df

,date,LE,country_code,LP,LUR,labour_force
0,2024-12-31,159.103,USA,337.765,4.920,167.335928
1,2023-12-31,159.375,USA,335.537,3.832,165.725605
2,2022-12-31,158.297,USA,333.530,3.642,164.280081
3,2021-12-31,152.586,USA,332.314,5.367,161.239737
4,2020-12-31,147.813,USA,331.257,8.092,160.827131
...,...,...,...,...,...,...
151,2003-12-31,9.392,AUS,19.827,5.942,9.985328
152,2002-12-31,9.185,AUS,19.605,6.358,9.808633
153,2001-12-31,9.016,AUS,19.386,6.775,9.671226
154,2000-12-31,8.899,AUS,19.141,6.292,9.496521


Q5. Create a new column "LFPR"

    LFPR = Labor Force / Population * 100


In [21]:
combined_df['LFPR'] = combined_df['labour_force']/combined_df['LP']*100
combined_df

,date,LE,country_code,LP,LUR,labour_force,LFPR
0,2024-12-31,159.103,USA,337.765,4.920,167.335928,49.542116
1,2023-12-31,159.375,USA,335.537,3.832,165.725605,49.391157
2,2022-12-31,158.297,USA,333.530,3.642,164.280081,49.254964
3,2021-12-31,152.586,USA,332.314,5.367,161.239737,48.520296
4,2020-12-31,147.813,USA,331.257,8.092,160.827131,48.550561
...,...,...,...,...,...,...,...
151,2003-12-31,9.392,AUS,19.827,5.942,9.985328,50.362275
152,2002-12-31,9.185,AUS,19.605,6.358,9.808633,50.031282
153,2001-12-31,9.016,AUS,19.386,6.775,9.671226,49.887679
154,2000-12-31,8.899,AUS,19.141,6.292,9.496521,49.613506


Q6. Show the mean LFPR for each country betwen 1999 to 2024, in decending order

In [22]:
combined_df.groupby('country_code')['LFPR'].mean().sort_values(ascending=False)

,LFPR
country_code,
CAN,53.623514
JPN,53.047032
AUS,51.981957
GBR,50.381297
USA,49.721489
FRA,46.267806
